## Persistent Landing - Delta Lake

The aim of this notebook is to convert all structure data we have into parquet files, in our case, it would be CSV files.

**Importing Useful Libraries**

In [ ]:
import os
import boto3
import duckdb
import pandas as pd
from datetime import datetime
from dotenv import load_dotenv
from deltalake import DeltaTable, write_deltalake
import polars as pl

# Load .env file
load_dotenv()

# Read environment variables
endpoint = os.getenv("MINIO_ENDPOINT")
access_key = os.getenv("MINIO_ACCESS_KEY")
secret_key = os.getenv("MINIO_SECRET_KEY")

In [ ]:
# Setup S3 client for MinIO (MinIO implements Amazon S3 API)
s3 = boto3.client(
    "s3",
    endpoint_url=endpoint, # MinIO API endpoint
    aws_access_key_id=access_key, # User name
    aws_secret_access_key=secret_key, # Password
)

In [ ]:
# Connect to DuckDB and configure S3 secret for MinIO
con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute(f"""
CREATE OR REPLACE SECRET secret (
    TYPE s3,
    PROVIDER config,
    ENDPOINT '{endpoint.replace("http://", "").replace("https://", "")}',
    KEY_ID '{access_key}',
    SECRET '{secret_key}',
    URL_STYLE 'path',
    USE_SSL false
);
""")

This function performs **zero-schema data integrity verification** between a source CSV file and a Delta table.

It is designed to validate:

- Row consistency
- Column consistency
- Content-level equality (order-agnostic)

The verification does **not rely on schema definitions**, making it suitable for flexible or evolving datasets.


In [ ]:
def verify_data_integrity(original_csv, delta_table_path, db_connection):
    """
    Zero-Schema Data Integrity Verification for Notebooks.
    """
    # Initialize connection
    con = db_connection

    print("--- Starting Verification ---")

    # Load Delta extension
    try:
        con.execute("INSTALL delta; LOAD delta;")
    except Exception:
        pass

    try:
        # STEP 1: Structural Validation (Row/Col counts)
        structure_sql = f"""
            SELECT
                (SELECT count(*) FROM read_csv_auto('{original_csv}')) as csv_rows,
                (SELECT count(*) FROM delta_scan('{delta_table_path}')) as delta_rows,
                (SELECT count(*) FROM (DESCRIBE SELECT * FROM read_csv_auto('{original_csv}'))) as csv_cols,
                (SELECT count(*) FROM (DESCRIBE SELECT * FROM delta_scan('{delta_table_path}'))) as delta_cols
        """
        csv_rows, delta_rows, csv_cols, delta_cols = con.execute(structure_sql).fetchone()

        if csv_rows != delta_rows or csv_cols != delta_cols:
            print(f"❌ Mismatch! CSV: {csv_rows}r/{csv_cols}c, Delta: {delta_rows}r/{delta_cols}c")
            return False

        # STEP 2: Content Inspection (Order-agnostic fingerprinting)
        # Cast to VARCHAR to avoid type-storage conflicts
        fingerprint_sql = f"""
            WITH csv_sig AS (
                SELECT bit_xor(hash(coalesce(columns(*)::VARCHAR, 'NULL'))) as sign
                FROM read_csv_auto('{original_csv}')
            ),
            delta_sig AS (
                SELECT bit_xor(hash(coalesce(columns(*)::VARCHAR, 'NULL'))) as sign
                FROM delta_scan('{delta_table_path}')
            )
            SELECT csv_sig.sign == delta_sig.sign FROM csv_sig, delta_sig
        """

        is_identical = con.execute(fingerprint_sql).fetchone()[0]

        if not is_identical:
            print("❌ Integrity Failed: Fingerprints do not match.")
            return False

        print(f"✅ Passed: {csv_rows} rows and {csv_cols} columns are identical.")
        return True

    except Exception as e:
        print(f"⚠️ Error during verification: {e}")
        return False

To build a high-performance "**Bronze Layer**" (this is simulated by the sub-bucket `csv-delta-lake`), this pipeline automates the conversion of raw CSV files into versioned **Delta Tables** by leveraging **DuckDB** for high-speed S3 streaming and Parquet conversion, followed by **Polars** to finalize the ACID-compliant Delta format. This transition optimizes the data lake for columnar performance and transactional integrity (via the `_delta_log`), while maintaining a clean environment by automatically purging intermediate Parquet files and providing full compatibility with S3-compatible storage through specialized `storage_options`.

In [ ]:
storage_options = {
    "AWS_ACCESS_KEY_ID": access_key,
    "AWS_SECRET_ACCESS_KEY": secret_key,
    "AWS_ENDPOINT_URL": endpoint,
    "AWS_S3_ALLOW_UNSAFE_RENAME": "true",
    "AWS_S3_ADDRESSING_STYLE": "path",
    "AWS_ALLOW_HTTP": "true",
    "region": "us-east-1"
}

def ingest_csv_to_delta(bucket, csv_prefix="persistent-landing/structured/"):
    """
    Unified pipeline: 
    1. DuckDB reads CSV and converts to a temporary Parquet.
    2. Polars/DeltaLake reads that Parquet and writes it as a versioned Delta Table.
    3. Cleans up the temporary Parquet.
    """
    # Initialize DuckDB S3 access
    con.execute("INSTALL httpfs; LOAD httpfs;")
    
    paginator = s3.get_paginator("list_objects_v2")
    delta_base_prefix = "persistent-landing/structured/raw"

    for page in paginator.paginate(Bucket=bucket, Prefix=csv_prefix):
        for obj in page.get("Contents", []):
            src_key = obj["Key"]
            
            # Skip directories
            if obj['Size'] == 0 and src_key.endswith("/"):
                continue

            # Extract base name (e.g., 'users' from 'path/to/users.csv')
            file_base = os.path.splitext(os.path.basename(src_key))[0]
            
            # Path Definitions
            s3_csv_path = f"s3://{bucket}/{src_key}"
            temp_parquet_path = f"s3://{bucket}/{delta_base_prefix}/{file_base}_temp.parquet"
            target_delta_folder = f"s3://{bucket}/{delta_base_prefix}/{file_base}/"

            print(f"🚀 Processing: {file_base}...")

            try:
                # --- Phase 1: DuckDB (CSV to Parquet) ---
                con.execute(f"""
                    COPY (SELECT * FROM read_csv_auto('{s3_csv_path}')) 
                    TO '{temp_parquet_path}' (FORMAT PARQUET);
                """)
                print(f"  └─ DuckDB: CSV converted to temporary Parquet.")

                # --- Phase 2: Delta Lake (Parquet to Delta) ---
                df = pl.read_parquet(temp_parquet_path, storage_options=storage_options)
                
                write_deltalake(
                    target_delta_folder,
                    df,
                    mode="overwrite",
                    storage_options=storage_options
                )
                print(f"  └─ Delta: Version 0 created at {target_delta_folder}")

                # --- Phase 3: Cleanup ---
                # Delete the temporary parquet file
                s3.delete_object(Bucket=bucket, Key=f"{delta_base_prefix}/{file_base}_temp.parquet")
                print(f"✅ Successfully finalized {file_base}.")
                if verify_data_integrity(s3_csv_path,target_delta_folder,con):
                    s3.delete_object(Bucket=bucket, Key=f"{delta_base_prefix}/{file_base}.csv")
                    print(f"✅ Successfully deleted csv file {s3_csv_path}.")

                else:
                    raise ValueError("Verification process crashed")


            except Exception as e:
                print(f"❌ Failed to process {file_base}: {e}")

In [ ]:
# Convert csv files into parquet files
ingest_csv_to_delta("landing-zone")

We can now try querying over these parquet filed.

In [ ]:
print("🔎 Reading first 10 rows of co2-emission.parquet via DuckDB:")
table_path = "s3://landing-zone/persistent-landing/structured/co2-emission*/part*.parquet"
df_view = con.execute(f"SELECT * FROM read_parquet('{table_path}') LIMIT 10").df()
df_view